In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
df = pd.read_csv("/content/medical_nlp_dataset.csv")

In [5]:
print(df)
X = df['symptoms']
Y = df["condition"]
print(X)

                         symptoms       condition
0            fever cough headache             flu
1  chest pain shortness of breath     heart_issue
2             sneezing runny nose            cold
3      high temperature body pain     viral_fever
4           stomach pain vomiting  food_poisoning
5               sore throat fever       infection
6               skin rash itching         allergy
7             joint pain swelling       arthritis
8               fatigue dizziness          anemia
9         blurred vision headache        migraine
0              fever cough headache
1    chest pain shortness of breath
2               sneezing runny nose
3        high temperature body pain
4             stomach pain vomiting
5                 sore throat fever
6                 skin rash itching
7               joint pain swelling
8                 fatigue dizziness
9           blurred vision headache
Name: symptoms, dtype: object


In [7]:
encoder = LabelEncoder()
Y = encoder.fit_transform(Y)

In [9]:
max_words = 1000
max_len = 20

tokenizer = Tokenizer(num_words = max_words)
tokenizer.fit_on_texts(X)
sequences = tokenizer.texts_to_sequences(X)

In [10]:
X = pad_sequences(sequences, maxlen = max_len)

In [22]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2,random_state=42)

In [23]:
model = Sequential([
    Embedding(input_dim=max_words, output_dim=64, input_length=max_len),
    LSTM(64),
    Dense(len(set(Y)), activation='softmax')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [24]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [25]:
model.fit(
    X_train,
    Y_train,
    epochs=20,
    batch_size=2,
    validation_data=(X_test, Y_test)
)

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 149ms/step - accuracy: 0.1250 - loss: 2.3106 - val_accuracy: 0.0000e+00 - val_loss: 2.3294
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.3750 - loss: 2.2880 - val_accuracy: 0.0000e+00 - val_loss: 2.3654
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3750 - loss: 2.2676 - val_accuracy: 0.0000e+00 - val_loss: 2.4034
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.2500 - loss: 2.2503 - val_accuracy: 0.0000e+00 - val_loss: 2.4543
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.3750 - loss: 2.2289 - val_accuracy: 0.0000e+00 - val_loss: 2.5304
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5000 - loss: 2.2040 - val_accuracy: 0.0000e+00 - val_loss: 2.6499
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3750 - loss: 2.1703 - val_accuracy: 0.0000e+00 - val_loss: 2.8271
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3750 - loss: 2.1272 - val_accurac

In [27]:
loss, accuracy = model.evaluate(X_test, Y_test)
print("Accuracy", accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.0000e+00 - loss: 5.6147
Accuracy 0.0


In [29]:
new_symptom = ["stomach pain vomiting"]

new_seq = tokenizer.texts_to_sequences(new_symptom)
new_pad = pad_sequences(new_seq, maxlen=max_len)

prediction = model.predict(new_pad)

predicted_class = prediction.argmax()

print("Predicted Disease:",
      encoder.inverse_transform([predicted_class])[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Predicted Disease: food_poisoning
